<img src="UKKA_UKRDC.svg" width="500" />

# KRT Demographics
Interactive visualization of UKRDC data by centre, variable, and dialect type. Run with voila to turn into a simple dashboard app.

In [1]:
from pathlib import Path
import pandas as pd

# Load data
path = Path.cwd()
file_path = path.parent / ".do_not_commit" / "krt_demog_ukrdc_live_2024.csv"

try:
    df = pd.read_csv(file_path)
except FileNotFoundError:
    print(f"Error: Could not find {file_path}")
    df = None

In [ ]:


import plotly.graph_objects as go
from ipywidgets import widgets
from IPython.display import display



if df is not None:
    # Get unique values for dropdowns
    variable2_options = sorted(df['variable2'].unique())
    dialtplt_options = ['all'] + sorted([x for x in df['dialtplt'].unique() if pd.notna(x)])
    centre_options = sorted(df['centre'].unique())
    
    # Create widgets
    variable2_dropdown = widgets.Dropdown(
        options=variable2_options,
        value=variable2_options[0] if variable2_options else None,
        description='Demographic Variable:',
        style={'description_width': '100px'}
    )
    
    dialtplt_dropdown = widgets.Dropdown(
        options=dialtplt_options,
        value='all',
        description='KRT Type:',
        style={'description_width': '100px'}
    )
    
    centre_dropdown = widgets.Dropdown(
        options=centre_options,
        value=centre_options[0] if centre_options else None,
        description='Centre:',
        style={'description_width': '100px'}
    )
    
    # Create FigureWidget
    fig_widget = go.FigureWidget()
    
    # Define update function
    def update_plot(variable2, dialtplt, centre):
        # Filter data
        filtered_df = df[df['variable2'] == variable2].copy()
        
        if dialtplt != 'all':
            filtered_df = filtered_df[filtered_df['dialtplt'] == dialtplt]
        
        filtered_df = filtered_df[filtered_df['centre'] == centre]
        
        if filtered_df.empty:
            fig_widget.data = []
            fig_widget.update_layout(title=f"No data available for {centre} - {variable2}")
            return
        
        # Aggregate data
        agg_df = filtered_df.groupby(['centre', 'quarter', 'year', 'variable'])['value'].sum().reset_index()
        
        # Create period column
        agg_df['period'] = agg_df['year'].astype(str) + ' Q' + agg_df['quarter'].astype(str)
        
        # Clear existing traces
        fig_widget.data = []
        
        # Add traces for each variable
        for var in agg_df['variable'].unique():
            var_data = agg_df[agg_df['variable'] == var]
            fig_widget.add_trace(
                go.Bar(
                    x=var_data['period'],
                    y=var_data['value'],
                    name=str(var),
                    hovertemplate='<b>%{x}</b><br>%{fullData.name}: %{y}<extra></extra>'
                )
            )
        
        # Update layout
        fig_widget.update_layout(
            title=f"{centre} - {variable2}",
            xaxis_title='Year / Quarter',
            yaxis_title='Count',
            barmode='stack',
            height=500,
            template='plotly_white',
            xaxis={'categoryorder': 'category ascending'},
            hovermode='x unified'
        )
    
    # Connect dropdown changes
    def on_change(change):
        update_plot(variable2_dropdown.value, dialtplt_dropdown.value, centre_dropdown.value)
    
    variable2_dropdown.observe(on_change, names='value')
    dialtplt_dropdown.observe(on_change, names='value')
    centre_dropdown.observe(on_change, names='value')
    
    # Display controls and figure
    controls = widgets.VBox([variable2_dropdown, dialtplt_dropdown, centre_dropdown])

    # Trigger initial plot
    update_plot(variable2_dropdown.value, dialtplt_dropdown.value, centre_dropdown.value)

    display(controls)
    display(fig_widget)
    

else:
    print("Unable to load data. Please check the file path.")

FigureWidget({
    'data': [{'hovertemplate': '<b>%{x}</b><br>%{fullData.name}: %{y}<extra></extra>',
              'name': '18-34',
              'type': 'bar',
              'uid': '618faddd-4836-4b9b-967d-9166953d4799',
              'x': array(['2024 Q1'], dtype=object),
              'y': {'bdata': 'Vg==', 'dtype': 'i1'}},
             {'hovertemplate': '<b>%{x}</b><br>%{fullData.name}: %{y}<extra></extra>',
              'name': '35-54',
              'type': 'bar',
              'uid': '71d6a271-340b-48b8-b2ea-b91e2f923bb7',
              'x': array(['2024 Q1'], dtype=object),
              'y': {'bdata': 'RgE=', 'dtype': 'i2'}},
             {'hovertemplate': '<b>%{x}</b><br>%{fullData.name}: %{y}<extra></extra>',
              'name': '55-74',
              'type': 'bar',
              'uid': '5abaec52-af70-4322-becf-ff2ab94a2869',
              'x': array(['2024 Q1'], dtype=object),
              'y': {'bdata': 'YgI=', 'dtype': 'i2'}},
             {'hovertemplate': '<b>%{x}<